# Day 051 Solution — Local AI Chat App

Section 4, Day 1. Builds the `ChatApp` logic core, drives it headlessly against Ollama, then generates a real, runnable `app.py` Streamlit file. This notebook never launches a server — it verifies the logic and the generated app instead. Run the app with `streamlit run app.py`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import ollama


def init_session(state: dict) -> dict:
    """
    Idempotently initialise a Streamlit-style session_state dict.

    Streamlit reruns the WHOLE script top-to-bottom on every interaction, so
    initialisation must never overwrite existing data. Only set a key if absent.

    Ensures keys:
        'messages'  -> list of {'role', 'content'} dicts (starts empty)
        'settings'  -> {'model', 'temperature', 'system_prompt'}
    Returns the same dict, mutated in place.
    """
    if 'messages' not in state:
        state['messages'] = []
    if 'settings' not in state:
        state['settings'] = {
            'model': 'llama3.2',
            'temperature': 0.7,
            'system_prompt': 'You are a helpful assistant.',
        }
    return state


def add_message(state: dict, role: str, content: str) -> dict:
    """Append a {'role', 'content'} message to state['messages']; return it."""
    if role not in ('user', 'assistant', 'system'):
        raise ValueError(f'invalid role: {role!r}')
    msg = {'role': role, 'content': content}
    state['messages'].append(msg)
    return msg


def reset_messages(state: dict) -> None:
    """Clear the conversation but keep settings (a 'Clear chat' button)."""
    state['messages'] = []


def validate_user_input(text: str, max_chars: int = 2000) -> tuple[bool, str]:
    """
    Validate raw text from an st.chat_input / st.text_area widget before it is
    sent to the model.

    Returns (is_valid, result):
      - empty/whitespace : (False, 'Please enter a message.')
      - too long         : (False, 'Message too long (max N chars).')
      - valid            : (True, cleaned_text)   # stripped
    """
    cleaned = text.strip()
    if not cleaned:
        return (False, 'Please enter a message.')
    if len(cleaned) > max_chars:
        return (False, f'Message too long (max {max_chars} chars).')
    return (True, cleaned)


def clamp(value: float, lo: float, hi: float) -> float:
    """Clamp a widget value into [lo, hi]. st.slider bounds live input, but a
    value restored from session_state or a URL param may be out of range."""
    return max(lo, min(hi, value))


def build_settings(model: str, temperature: float, system_prompt: str) -> dict:
    """
    Assemble a validated settings dict from sidebar widget values.
    - temperature clamped to [0.0, 1.0]
    - system_prompt stripped; empty falls back to a default
    """
    sp = system_prompt.strip() or 'You are a helpful assistant.'
    return {
        'model': model,
        'temperature': float(clamp(temperature, 0.0, 1.0)),
        'system_prompt': sp,
    }


def build_messages(state: dict, user_text: str) -> list:
    """
    Build the messages list for ollama.chat:
        [system_prompt] + prior conversation + new user turn.
    Reads the system prompt from state['settings']. Does NOT mutate state.
    """
    settings = state.get('settings', {})
    system_prompt = settings.get('system_prompt', 'You are a helpful assistant.')
    messages = [{'role': 'system', 'content': system_prompt}]
    messages.extend(state.get('messages', []))
    messages.append({'role': 'user', 'content': user_text})
    return messages


def chat_with_history(state: dict, user_text: str, model: str = 'llama3.2') -> str:
    """
    Send the full conversation to Ollama and return the assistant's reply.
    Reads temperature from state['settings']. Returns a fallback string if
    Ollama is unavailable so the app never crashes on a model error.
    """
    settings = state.get('settings', {})
    temperature = settings.get('temperature', 0.7)
    messages = build_messages(state, user_text)
    try:
        response = ollama.chat(
            model=model,
            messages=messages,
            options={'temperature': temperature},
        )
        return response['message']['content'].strip()
    except Exception as e:
        return f'[Model unavailable: {e}]' 


def format_transcript(messages: list) -> str:
    """
    Render the conversation as a plain-text transcript for st.download_button.
    One block per turn as 'ROLE: content'. System messages are skipped.
    """
    lines = []
    for m in messages:
        role = m.get('role', '')
        if role == 'system':
            continue
        lines.append(f"{role.upper()}: {m.get('content', '')}")
    return '\n\n'.join(lines)


def chat_stats(messages: list) -> dict:
    """
    Compute display metrics for st.metric widgets. System messages excluded.
    Returns: {'total', 'user', 'assistant', 'chars'}.
    """
    non_system = [m for m in messages if m.get('role') != 'system']
    user = sum(1 for m in non_system if m.get('role') == 'user')
    assistant = sum(1 for m in non_system if m.get('role') == 'assistant')
    chars = sum(len(m.get('content', '')) for m in non_system)
    return {
        'total': len(non_system),
        'user': user,
        'assistant': assistant,
        'chars': chars,
    }


class ChatApp:
    """
    The logic core of the Streamlit chat app. One instance is stored in
    st.session_state and reused across reruns. The Streamlit layer only calls
    these methods and renders their return values — no business logic in the UI.

    Usage (inside app.py):
        if 'app' not in st.session_state:
            st.session_state.app = ChatApp()
        app = st.session_state.app
        reply = app.send(prompt)          # on chat_input submit
        st.metric('Messages', app.stats()['total'])
    """

    def __init__(self, model: str = 'llama3.2',
                 system_prompt: str = 'You are a helpful assistant.'):
        self.state = {}
        init_session(self.state)
        self.state['settings']['model'] = model
        self.state['settings']['system_prompt'] = system_prompt
        self.model = model

    def send(self, user_text: str) -> str:
        """
        Validate -> append user turn -> call model -> append assistant turn.
        Returns the assistant reply, or a validation error string (in which
        case NOTHING is appended to the conversation).
        """
        ok, result = validate_user_input(user_text)
        if not ok:
            return result
        add_message(self.state, 'user', result)
        reply = chat_with_history(self.state, result, self.model)
        add_message(self.state, 'assistant', reply)
        return reply

    def stats(self) -> dict:
        return chat_stats(self.state['messages'])

    def transcript(self) -> str:
        return format_transcript(self.state['messages'])

    def reset(self) -> None:
        reset_messages(self.state)

## Step 1 — Drive the ChatApp Logic Headlessly

In [ ]:
app = ChatApp(system_prompt='You are a terse assistant. Answer briefly.')
print('Reply 1:', app.send('Give me one tip for writing clean Python.'))
print('Reply 2:', app.send('Summarise that in five words.'))
print()
print('Stats:', app.stats())
assert app.stats()['total'] == 4, 'expected 2 user + 2 assistant turns'

## Step 2 — Validation + Reset Behaviour

In [ ]:
err = app.send('   ')  # blank input
print('Blank input ->', err)
assert 'enter a message' in err.lower()
assert app.stats()['total'] == 4, 'blank input must not append'

print('\nTranscript so far:')
print(app.transcript())

app.reset()
assert app.stats()['total'] == 0
print('\nAfter reset, messages =', app.stats()['total'])

## Step 3 — Generate the Real Streamlit app.py

In [ ]:
from pathlib import Path

# The full Streamlit app source: the logic functions + ChatApp shown above,
# plus the UI layer. Embedded as a string so we can write it to a real file.
_APP_SRC = 'import streamlit as st\nimport ollama\n\n\ndef init_session(state: dict) -> dict:\n    """\n    Idempotently initialise a Streamlit-style session_state dict.\n\n    Streamlit reruns the WHOLE script top-to-bottom on every interaction, so\n    initialisation must never overwrite existing data. Only set a key if absent.\n\n    Ensures keys:\n        \'messages\'  -> list of {\'role\', \'content\'} dicts (starts empty)\n        \'settings\'  -> {\'model\', \'temperature\', \'system_prompt\'}\n    Returns the same dict, mutated in place.\n    """\n    if \'messages\' not in state:\n        state[\'messages\'] = []\n    if \'settings\' not in state:\n        state[\'settings\'] = {\n            \'model\': \'llama3.2\',\n            \'temperature\': 0.7,\n            \'system_prompt\': \'You are a helpful assistant.\',\n        }\n    return state\n\n\ndef add_message(state: dict, role: str, content: str) -> dict:\n    """Append a {\'role\', \'content\'} message to state[\'messages\']; return it."""\n    if role not in (\'user\', \'assistant\', \'system\'):\n        raise ValueError(f\'invalid role: {role!r}\')\n    msg = {\'role\': role, \'content\': content}\n    state[\'messages\'].append(msg)\n    return msg\n\n\ndef reset_messages(state: dict) -> None:\n    """Clear the conversation but keep settings (a \'Clear chat\' button)."""\n    state[\'messages\'] = []\n\n\ndef validate_user_input(text: str, max_chars: int = 2000) -> tuple[bool, str]:\n    """\n    Validate raw text from an st.chat_input / st.text_area widget before it is\n    sent to the model.\n\n    Returns (is_valid, result):\n      - empty/whitespace : (False, \'Please enter a message.\')\n      - too long         : (False, \'Message too long (max N chars).\')\n      - valid            : (True, cleaned_text)   # stripped\n    """\n    cleaned = text.strip()\n    if not cleaned:\n        return (False, \'Please enter a message.\')\n    if len(cleaned) > max_chars:\n        return (False, f\'Message too long (max {max_chars} chars).\')\n    return (True, cleaned)\n\n\ndef clamp(value: float, lo: float, hi: float) -> float:\n    """Clamp a widget value into [lo, hi]. st.slider bounds live input, but a\n    value restored from session_state or a URL param may be out of range."""\n    return max(lo, min(hi, value))\n\n\ndef build_settings(model: str, temperature: float, system_prompt: str) -> dict:\n    """\n    Assemble a validated settings dict from sidebar widget values.\n    - temperature clamped to [0.0, 1.0]\n    - system_prompt stripped; empty falls back to a default\n    """\n    sp = system_prompt.strip() or \'You are a helpful assistant.\'\n    return {\n        \'model\': model,\n        \'temperature\': float(clamp(temperature, 0.0, 1.0)),\n        \'system_prompt\': sp,\n    }\n\n\ndef build_messages(state: dict, user_text: str) -> list:\n    """\n    Build the messages list for ollama.chat:\n        [system_prompt] + prior conversation + new user turn.\n    Reads the system prompt from state[\'settings\']. Does NOT mutate state.\n    """\n    settings = state.get(\'settings\', {})\n    system_prompt = settings.get(\'system_prompt\', \'You are a helpful assistant.\')\n    messages = [{\'role\': \'system\', \'content\': system_prompt}]\n    messages.extend(state.get(\'messages\', []))\n    messages.append({\'role\': \'user\', \'content\': user_text})\n    return messages\n\n\ndef chat_with_history(state: dict, user_text: str, model: str = \'llama3.2\') -> str:\n    """\n    Send the full conversation to Ollama and return the assistant\'s reply.\n    Reads temperature from state[\'settings\']. Returns a fallback string if\n    Ollama is unavailable so the app never crashes on a model error.\n    """\n    settings = state.get(\'settings\', {})\n    temperature = settings.get(\'temperature\', 0.7)\n    messages = build_messages(state, user_text)\n    try:\n        response = ollama.chat(\n            model=model,\n            messages=messages,\n            options={\'temperature\': temperature},\n        )\n        return response[\'message\'][\'content\'].strip()\n    except Exception as e:\n        return f\'[Model unavailable: {e}]\' \n\n\ndef format_transcript(messages: list) -> str:\n    """\n    Render the conversation as a plain-text transcript for st.download_button.\n    One block per turn as \'ROLE: content\'. System messages are skipped.\n    """\n    lines = []\n    for m in messages:\n        role = m.get(\'role\', \'\')\n        if role == \'system\':\n            continue\n        lines.append(f"{role.upper()}: {m.get(\'content\', \'\')}")\n    return \'\\n\\n\'.join(lines)\n\n\ndef chat_stats(messages: list) -> dict:\n    """\n    Compute display metrics for st.metric widgets. System messages excluded.\n    Returns: {\'total\', \'user\', \'assistant\', \'chars\'}.\n    """\n    non_system = [m for m in messages if m.get(\'role\') != \'system\']\n    user = sum(1 for m in non_system if m.get(\'role\') == \'user\')\n    assistant = sum(1 for m in non_system if m.get(\'role\') == \'assistant\')\n    chars = sum(len(m.get(\'content\', \'\')) for m in non_system)\n    return {\n        \'total\': len(non_system),\n        \'user\': user,\n        \'assistant\': assistant,\n        \'chars\': chars,\n    }\n\n\nclass ChatApp:\n    """\n    The logic core of the Streamlit chat app. One instance is stored in\n    st.session_state and reused across reruns. The Streamlit layer only calls\n    these methods and renders their return values — no business logic in the UI.\n\n    Usage (inside app.py):\n        if \'app\' not in st.session_state:\n            st.session_state.app = ChatApp()\n        app = st.session_state.app\n        reply = app.send(prompt)          # on chat_input submit\n        st.metric(\'Messages\', app.stats()[\'total\'])\n    """\n\n    def __init__(self, model: str = \'llama3.2\',\n                 system_prompt: str = \'You are a helpful assistant.\'):\n        self.state = {}\n        init_session(self.state)\n        self.state[\'settings\'][\'model\'] = model\n        self.state[\'settings\'][\'system_prompt\'] = system_prompt\n        self.model = model\n\n    def send(self, user_text: str) -> str:\n        """\n        Validate -> append user turn -> call model -> append assistant turn.\n        Returns the assistant reply, or a validation error string (in which\n        case NOTHING is appended to the conversation).\n        """\n        ok, result = validate_user_input(user_text)\n        if not ok:\n            return result\n        add_message(self.state, \'user\', result)\n        reply = chat_with_history(self.state, result, self.model)\n        add_message(self.state, \'assistant\', reply)\n        return reply\n\n    def stats(self) -> dict:\n        return chat_stats(self.state[\'messages\'])\n\n    def transcript(self) -> str:\n        return format_transcript(self.state[\'messages\'])\n\n    def reset(self) -> None:\n        reset_messages(self.state)\n\n\n# ---- Streamlit UI (runs only under: streamlit run app.py) ----\nst.set_page_config(page_title="Local AI Chat", page_icon="💬")\nst.title("💬 Local AI Chat")\n\n# One ChatApp instance per browser session, persisted across reruns.\nif "app" not in st.session_state:\n    st.session_state.app = ChatApp()\napp = st.session_state.app\n\n# Sidebar: settings + live stats + controls.\nwith st.sidebar:\n    st.header("Settings")\n    temp = st.slider("Temperature", 0.0, 1.0, 0.7, 0.1)\n    sysp = st.text_area("System prompt", app.state["settings"]["system_prompt"])\n    app.state["settings"] = build_settings(app.model, temp, sysp)\n\n    st.header("Stats")\n    s = app.stats()\n    c1, c2 = st.columns(2)\n    c1.metric("Messages", s["total"])\n    c2.metric("Characters", s["chars"])\n\n    if st.button("Clear chat"):\n        app.reset()\n        st.rerun()\n\n    st.download_button("Download transcript", app.transcript(), "chat.txt")\n\n# Replay the whole conversation from session state.\nfor m in app.state["messages"]:\n    with st.chat_message(m["role"]):\n        st.markdown(m["content"])\n\n# Handle a new user turn.\nprompt = st.chat_input("Type a message...")\nif prompt:\n    with st.spinner("Thinking..."):\n        app.send(prompt)          # appends user + assistant to session state\n    st.rerun()                    # rerun so the replay loop renders them\n'


def write_streamlit_app(path: str = 'app.py') -> str:
    """Write the self-contained Streamlit app to `path` and return the path."""
    Path(path).write_text(_APP_SRC, encoding='utf-8')
    return path

In [ ]:
path = write_streamlit_app('app.py')
src = open(path, encoding='utf-8').read()
print(f'Wrote {path} ({len(src)} chars)')

# Verify the generated app is well-formed
assert 'import streamlit as st' in src
assert 'class ChatApp' in src
assert 'st.session_state' in src
assert 'st.chat_input' in src
compile(src, 'app.py', 'exec')  # must be valid Python
print('app.py verified: imports streamlit, defines ChatApp, uses chat UI, compiles.')

## Step 4 — Preview the Generated UI Section

In [ ]:
ui_start = src.index('# ---- Streamlit UI')
print(src[ui_start:ui_start + 700])

print('\nTo launch the app, run in a terminal:')
print('    streamlit run app.py')
print('\nDay 51 — Local AI Chat App complete! 🎉')